In [1]:
import pandas as pd
import os

In [2]:
os.makedirs("batches", exist_ok = True)

orders = pd.read_csv(r"C:\Users\sy2825\Downloads\OLIST DATASET\olist_orders_dataset.csv")
order_items = pd.read_csv(r"C:\Users\sy2825\Downloads\OLIST DATASET\olist_order_items_dataset.csv")
order_reviews = pd.read_csv(r"C:\Users\sy2825\Downloads\OLIST DATASET\olist_order_reviews_dataset.csv")
order_payments = pd.read_csv(r"C:\Users\sy2825\Downloads\OLIST DATASET\olist_order_payments_dataset.csv")
customers = pd.read_csv(r"C:\Users\sy2825\Downloads\OLIST DATASET\olist_customers_dataset.csv")
products = pd.read_csv(r"C:\Users\sy2825\Downloads\OLIST DATASET\olist_products_dataset.csv")
product_category = pd.read_csv(r"C:\Users\sy2825\Downloads\OLIST DATASET\product_category_name_translation.csv")
sellers = pd.read_csv(r"C:\Users\sy2825\Downloads\OLIST DATASET\olist_sellers_dataset.csv")
geolocation = pd.read_csv(r"C:\Users\sy2825\Downloads\OLIST DATASET\olist_geolocation_dataset.csv")


In [3]:
order_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72


In [4]:
# parsing the date column
# create a customer month-year column in orders dataframe
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["year_month"] = orders["order_purchase_timestamp"].dt.to_period("M")


# loop through each month and save all related tables filterd to that month
for period, orders_batch in orders.groupby("year_month"):
    """
    create a folder per month
    period: holds the current month label
    orders_batch: holds all the rows belonging to that month as a df
    """
    folder = f"batches/{period}"
    os.makedirs(folder, exist_ok=True)
    """
    create folder path using the period batches
    """
    order_ids = orders_batch["order_id"].unique()
    """
    extract all unique order_id - this will be used to filter every other table
    """
    orders_batch.drop(columns = "year_month").to_csv(f"{folder}/orders-{period}.csv", index = False)
    """
    save monthly orders to a parquet file - drop the column yer_month as this was created for partitioning purposes
    """
    
    order_items[order_items["order_id"].isin(order_ids)].to_csv(f"{folder}/order_items-{period}.csv", index = False)
    
    """
    filter the full order items df to only keep rows where order_id appears in fact table - orders
    """
    order_reviews[order_reviews["order_id"].isin(order_ids)].to_csv(f"{folder}/order_reviews-{period}.csv", index = False)
    """
    filter the full order reviews df to only keep rows where order_id appears in fact table - orders
    """
    order_payments[order_payments["order_id"].isin(order_ids)].to_csv(f"{folder}/order_payments-{period}.csv", index = False)
    """
    filter the full order payments df to only keep rows where order_id appears in fact table - orders
    """
    
    customer_ids = orders_batch["customer_id"].unique()
    customers[customers["customer_id"].isin(customer_ids)].to_csv(f"{folder}/customers-{period}.csv", index = False)
    
    """
    customers are linked to orders via customer_id - first, extract unique customer id then link to orders_batch
    """
    
    batch_items = order_items[order_items["order_id"].isin(order_ids)]
    product_ids = batch_items["product_id"].unique()
    seller_ids = batch_items["seller_id"].unique()
    """
    products and sellers are not directly linked to orders - both are linked to order_items
    first, we filter order_items fo the batch(order_id) stored in order_item
    then use outcome to extract product_id and seller_id
    """

    products[products["product_id"].isin(product_ids)].to_csv(f"{folder}/products-{period}.csv", index = False)

    sellers[sellers["seller_id"].isin(seller_ids)].to_csv(f"{folder}/sellers-{period}.csv", index = False)

    geolocation.to_csv(f"{folder}/geolocation.csv", index = False)

    print(f"Batch {period} saved = {len(orders_batch)} orders")
    
    
    
        
                                                         

    

Batch 2016-09 saved = 4 orders
Batch 2016-10 saved = 324 orders
Batch 2016-12 saved = 1 orders
Batch 2017-01 saved = 800 orders
Batch 2017-02 saved = 1780 orders
Batch 2017-03 saved = 2682 orders
Batch 2017-04 saved = 2404 orders
Batch 2017-05 saved = 3700 orders
Batch 2017-06 saved = 3245 orders
Batch 2017-07 saved = 4026 orders
Batch 2017-08 saved = 4331 orders
Batch 2017-09 saved = 4285 orders
Batch 2017-10 saved = 4631 orders
Batch 2017-11 saved = 7544 orders
Batch 2017-12 saved = 5673 orders
Batch 2018-01 saved = 7269 orders
Batch 2018-02 saved = 6728 orders
Batch 2018-03 saved = 7211 orders
Batch 2018-04 saved = 6939 orders
Batch 2018-05 saved = 6873 orders
Batch 2018-06 saved = 6167 orders
Batch 2018-07 saved = 6292 orders
Batch 2018-08 saved = 6512 orders
Batch 2018-09 saved = 16 orders
Batch 2018-10 saved = 4 orders
